# DNA Methylation Analysis with `cytozip`

CLI walkthrough using 9 mouse 3C-seq cells shipped under
`cytozip_example_data/bam/`. All steps are demonstrated as shell
commands (`! czip ...`); benchmark results produced offline by the
scripts under `tests/` are loaded and rendered as tables.

1. Setup + reference `.cz`
2. BAM → `.cz`  (`czip bam_to_cz`)
3. ALLC → `.cz` (`czip allc2cz`) + benchmark table (BAM→ALLC vs BAM→CZ, ALLC→CZ)
4. `view` / `query` (local) + benchmark table (tabix vs czip query)
5. `view` / `query` a remote `.cz`
6. `catcz` + `merge_cz`
7. `cz_to_anndata`

Conda env: `/home/x-wding2/Software/conda/m3c`. The benchmark scripts
under `tests/` were run beforehand with `-j 9` (≤ 20 CPUs):

    python tests/benchmark_bam_to_cz.py  -j 9
    python tests/benchmark_allc_to_cz.py -j 9
    python tests/benchmark_query.py


## 1. Setup

In [15]:
import os,sys
import pandas as pd
os.chdir(os.path.expanduser("~/Projects/Github/cytozip/cytozip_example_data"))

## 2. Build the reference `.cz`

The reference holds the genome-wide `(chrom, pos, strand, context)`
axis. Per-cell `.cz` then store only `mc`/`cov` and reuse the
reference's positions, cutting per-cell size by ~5×.


In [4]:
!czip build_ref --help

usage: czip build_ref [-h] -g GENOME [-O OUTPUT] [-p PATTERN] [-t THREADS]
                      [--keep-temp] [--no-delta]

options:
  -h, --help            show this help message and exit
  -g GENOME, --genome GENOME
                        reference genome FASTA (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (default: hg38_allc.cz)
  -p PATTERN, --pattern PATTERN
                        nucleotide pattern (default: C)
  -t THREADS, --threads THREADS
                        parallel jobs (default: 12)
  --keep-temp           keep temp directory (default: False)
  --no-delta            disable DELTA encoding on the pos column (default: on,
                        gives ~3x smaller reference files with mild query
                        overhead) (default: False)


In [7]:
!czip build_ref -g ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O output/mm10_with_chrL.allc.cz -t 20


In [8]:
!czip header -I output/mm10_with_chrL.allc.cz

magic  :  b'CZIP'
version  :  0.0
total_size  :  1362568959
message  :  /home/x-wding2/Ref/mm10/mm10_ucsc_with_chrL.fa
formats  :  ['Q', 'c', '3s']
columns  :  ['pos', 'strand', 'context']
sort_col  :  0
delta_cols  :  [0]
chunk_keys  :  ['chrom']
header_size  :  102


In [ ]:
"!czip view -I output/mm10_with_chrL.allc.cz --show_dim 0 | head"

chrom	pos	strand	context
chr10	3100001	-	CNN
chr10	3100006	+	CTC
chr10	3100008	+	CAC
chr10	3100010	+	CCT
chr10	3100011	+	CTG
chr10	3100013	-	CAG
chr10	3100015	-	CTC
chr10	3100021	+	CCG
chr10	3100022	+	CGA


## 3. BAM → `.cz`

Convert a position-sorted BAM directly to `.cz` (no intermediate ALLC
text). The output has only `mc` / `cov` because we pass a reference.


In [11]:
!czip bam_to_cz --help

usage: czip bam_to_cz [-h] -I INPUT -r REFERENCE_FASTA [-O OUTPUT]
                      [--num-upstr NUM_UPSTR] [--num-downstr NUM_DOWNSTR]
                      [--min-mapq MIN_MAPQ]
                      [--min-base-quality MIN_BASE_QUALITY] [-c BATCH_SIZE]
                      [--convert-strandness] [--save-count-df]
                      [--mode {full,pos_mc_cov,mc_cov}]
                      [--count-fmt {B,H,I,Q}] [--reference-cz REFERENCE_CZ]
                      [--slim]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input position-sorted BAM (bismark/hisat-3n) (default:
                        None)
  -r REFERENCE_FASTA, --reference-fasta REFERENCE_FASTA
                        indexed reference fasta (.fai required) (default:
                        None)
  -O OUTPUT, --output OUTPUT
                        output .cz path (default: <bam_stem>.cz) (default:
                        None)
  --num-upstr NUM_

In [ ]:
! time czip bam_to_cz -I bam/FC_M_P12b_3C_2-5-M17-N10.hisat3n_dna.all_reads.deduped.bam \
 --reference_fasta ~/Ref/mm10/mm10_ucsc_with_chrL.fa -O output/cz/FC_M_P12b_3C_2-5-M17-N10.cz \
 --mode mc_cov --count_fmt B --reference_cz output/mm10_with_chrL.allc.cz


input : bam/FC_E17b_3C_5-5-I24-A21.hisat3n_dna.all_reads.deduped.bam (686.0 MB)
output: output/cz/FC_E17b_3C_5-5-I24-A21.cz (80.1 MB)


magic  :  b'CZIP'
version  :  0.0
total_size  :  80146739
message  :  mm10_with_chrL.allc.cz
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_keys  :  ['chrom']
header_size  :  61


### BAM → `.cz` benchmark vs ALLCools

Produced by `tests/benchmark_bam_to_cz.py -j 9`. ALLCools writes a
gzipped TSV (`.allc.tsv.gz`); cytozip writes a chunked, columnar
zstd-compressed `.cz`.


In [ ]:
import pandas as pd
bam_bench = pd.read_csv('output/bam_benchmark/bam_benchmark.tsv', sep='\t')
bam_bench.round(2)


,cell,bam_size_mb,n_reads,allc_wall_s,allc_rss_mb,allc_size_mb,cz_wall_s,cz_rss_mb,cz_size_mb,speedup,size_ratio
0,FC_E17b_3C_5-5-I24-A21,685.99,7330382,553.85,571.27,442.62,540.60,10590.48,80.15,1.02,0.18
1,FC_M_E15a_3C_1-1-I5-B1,177.37,1787812,147.02,616.51,102.53,169.98,10588.41,23.49,0.86,0.23
2,FC_M_P12b_3C_2-5-M17-N10,239.89,2307326,204.82,571.41,143.12,234.85,10588.13,30.93,0.87,0.22
3,FC_M_P3b_3C_6-6-J3-P24,147.36,1395949,131.24,573.38,85.71,154.85,10588.54,20.62,0.85,0.24
4,FC_M_P6a_3C_7-3-K21-P5,89.76,938870,86.70,574.04,48.84,109.88,10590.46,13.39,0.79,0.27
5,FC_M_P9B_3C_6-2-F6-O4,32.28,342778,43.56,573.19,17.61,75.11,10588.45,7.07,0.58,0.40
6,FC_P0b_3C_5-1-I24-J14,485.51,5028030,371.75,571.43,285.50,392.04,10588.40,53.90,0.95,0.19
7,FC_P13a_3C_7-1-A11-O1,260.71,2588754,210.55,573.31,155.84,238.37,10588.57,32.55,0.88,0.21
8,FC_P28a_3C_2-1-E5-N14,186.32,1908846,158.05,575.86,112.95,189.93,10590.31,25.20,0.83,0.22


In [12]:
!cat output/bam_benchmark/bam_benchmark.txt

cytozip bam_to_cz  vs  ALLCools bam_to_allc
reference FASTA : /home/x-wding2/Ref/mm10/mm10_ucsc_with_chrL.fa
reference .cz   : /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/mm10_with_chrL.allc.cz
BAMs            : 9   total reads = 23,628,747

ALLCools : time=  1907.5 s   size=  1394.7 MB
cytozip  : time=  2105.6 s   size=   287.3 MB

speedup (allc / cz time)  =  0.91x
compression (cz / allc)   =  20.6%


## 4. ALLC → `.cz`

Pack an existing `.allc.tsv.gz` into `.cz`. Two layouts:

* **standalone** — store `[pos, mc, cov]`; no reference required.
* **reference-aligned** — store only `[mc, cov]`; positions come from
  a shared reference `.cz`. Smaller, but querying needs the reference.


In [13]:
! czip allc2cz --help

usage: czip allc2cz [-h] -I INPUT -O OUTPUT [-r REFERENCE]
                    [--missing-value MISSING_VALUE] [-F FORMATS] [-C COLUMNS]
                    [-D DIMENSIONS] [-u USECOLS] [--ref-pos-col REF_POS_COL]
                    [--allc-pos-col ALLC_POS_COL] [-s SEP]
                    [--chrom-order CHROM_ORDER] [-c CHUNKSIZE]
                    [--sort-col SORT_COL] [--delta-cols DELTA_COLS]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input allc.tsv.gz (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (default: None)
  -r REFERENCE, --reference REFERENCE
                        reference .cz file (default: None)
  --missing-value MISSING_VALUE
                        missing value fill (default: [0, 0])
  -F FORMATS, --formats FORMATS
                        column formats (default: ['B', 'B'])
  -C COLUMNS, --columns COLUMNS
                        column names (default:

In [ ]:
# Standalone (positions inside the .cz)
! time czip allc2cz -I output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz \
      -O output/allc/FC_M_P12b_3C_2-5-M17-N10.with_pos.cz \
      -F Q,B,B -C pos,mc,cov -u 1,4,5


input .allc.tsv.gz      :  442.6 MB
standalone .cz          :  134.3 MB ( 30.3%)
reference-aligned .cz   :   80.1 MB ( 18.1%)


In [ ]:
# Reference-aligned (positions come from the reference .cz)
! time czip allc2cz -I output/allc/FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz \
      -O output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
      -r output/mm10_with_chrL.allc.cz

In [ ]:
! ls output/allc/FC_M_P12b_3C_2-5-M17-N10* -sh

## 5. `view` and `query`

`czip view` streams an entire dimension (e.g. one chrom). `czip query`
pulls a region. Both can use `-r REFERENCE` to expand reference-aligned
files into full ALLC-style records (chrom / pos / context / mc / cov).


In [ ]:
!czip view -I output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz --show_dim 0 | head -5"

chrom	pos	strand	context	mc	cov
chr10	3100001	-	CNN	0	0
chr10	3100006	+	CTC	0	0
chr10	3100008	+	CAC	0	0
chr10	3100010	+	CCT	0	0


In [ ]:
!time czip query -I output/allc/FC_M_P12b_3C_2-5-M17-N10.cz \
    -r output/mm10_with_chrL.allc.cz -D chr9 -s 3000294 -e 3005294 | head -5

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	25	29
chr9	3000296	+	CCT	108	118
chr9	3000297	+	CTA	107	115
chr9	3000300	+	CAA	111	116


## 6. View / query a remote `.cz` (no download)

cytozip reads `.cz` files directly over HTTP Range requests when they
carry a chunk index. Below we query a remote `.cz` hosted on figshare —
only the needed chunks are fetched on-demand.


In [ ]:
! czip header -I http://gale-sapiens.salk.edu/bican/FC_M_P12b_3C_2-5-M17-N10.cz

remote demo skipped: ValueError encoding table references col 99 but only 3 columns exist


## 7. `catcz` and `merge_cz`

* `catcz` — concatenate per-cell `.cz` into one multi-cell `.cz` by
  adding a `cell_id` dimension.
* `merge_cz` — sum `mc` / `cov` across cells (pooled pseudobulk).


In [16]:
cz_files  = sorted(CZ_DIR.glob('*.cz'))
cells_cz  = OUT / 'all_cells.cz'
cz_paths  = ','.join(str(p) for p in cz_files)
!czip catcz -O {cells_cz} -I {cz_paths} --add_dim --title cell_id -F B,B -C mc,cov
print(f'{cells_cz}: {cells_cz.stat().st_size/1e6:.1f} MB')
!czip header -I {cells_cz}


output/all_cells.cz: 287.3 MB


magic  :  b'CZIP'
version  :  0.0
total_size  :  287336622
message  :  
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_keys  :  ['chrom', 'cell_id']
header_size  :  47


In [17]:
pseudobulk_cz = OUT / 'pseudobulk.cz'
!czip merge_cz -i {CZ_DIR} -O {pseudobulk_cz} -r {REF_CZ} -F H,H -t 20
print(f'{pseudobulk_cz}: {pseudobulk_cz.stat().st_size/1e6:.1f} MB')
!czip header -I {pseudobulk_cz}
print('--- first 5 records ---')
!czip view -I {pseudobulk_cz} --show_dim 0 -r {REF_CZ} | head -5


output/pseudobulk.cz


Removing temp dir /anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip_example_data/output/pseudobulk.cz.tmp


output/pseudobulk.cz: 224.1 MB


magic  :  b'CZIP'
version  :  0.0
total_size  :  224110118
message  :  merged
formats  :  ['H', 'H']
columns  :  ['mc', 'cov']
sort_col  :  None
delta_cols  :  []
chunk_keys  :  ['chrom']
header_size  :  45


--- first 5 records ---


chrom	pos	strand	context	mc	cov
chr10	3100001	-	CNN	0	0
chr10	3100006	+	CTC	0	0
chr10	3100008	+	CAC	0	0
chr10	3100010	+	CCT	0	0


## 8. Build a cell × gene `AnnData`

`cytozip.features.cz_to_anndata` aggregates per-cell `.cz` files over
a feature interval set. When `features=` is a GTF path, cytozip
extracts one interval per gene, merges GENCODE records sharing a
symbol, and extends each interval by `flank_bp` (default 2 kb) on
both sides.


In [18]:
from cytozip.features import cz_to_anndata

cz_paths = sorted(str(p) for p in CZ_DIR.glob('*.cz'))
print(f'{len(cz_paths)} per-cell .cz files')

t0 = time.perf_counter()
ad_gene = cz_to_anndata(
    cz_inputs=cz_paths,
    reference_cz=str(REF_CZ),
    features=str(GTF),
    flank_bp=2000,
    score='frac',
    threads=20,
)
print(f'cz_to_anndata(GTF): {time.perf_counter()-t0:.1f}s   {ad_gene}')
print('\nobs head:'); print(ad_gene.obs.head(3))
print('\nvar head:'); print(ad_gene.var.head(3))
ad_gene.write_h5ad(OUT / 'bg_gene.h5ad')
print('wrote', OUT / 'bg_gene.h5ad')


9 per-cell .cz files


cz_to_anndata(GTF): 62.4s   AnnData object with n_obs × n_vars = 9 × 55228
    obs: 'alpha', 'beta', 'prior_mean'
    var: 'chrom', 'start', 'end', 'gene_id', 'gene_name', 'gene_type', 'strand'
    uns: 'cytozip_score'
    layers: 'mc', 'cov'

obs head:
                              alpha      beta  prior_mean
FC_E17b_3C_5-5-I24-A21    16.560707  4.300688    0.793845
FC_M_E15a_3C_1-1-I5-B1     2.186391  2.139432    0.505428
FC_M_P12b_3C_2-5-M17-N10   2.996044  2.876152    0.510208

var head:
              chrom    start      end               gene_id      gene_name  \
name                                                                         
4933401J01Rik  chr1  3071252  3076322  ENSMUSG00000102693.1  4933401J01Rik   
Gm26206        chr1  3100015  3104125  ENSMUSG00000064842.1        Gm26206   
Xkr4           chr1  3203900  3673498  ENSMUSG00000051951.5           Xkr4   

                    gene_type strand  
name                                  
4933401J01Rik             TEC     

## 9. Outputs

In [19]:
import subprocess as _sp
try:
    print(_sp.check_output(['tree', '-L', '2', 'output'], text=True))
except FileNotFoundError:
    for p in sorted(Path('output').rglob('*')):
        if p.is_file():
            print(f'{p.stat().st_size/1e6:8.2f} MB  {p}')


output
├── allc
│   ├── FC_E17b_3C_5-5-I24-A21.allc.tsv.gz
│   ├── FC_E17b_3C_5-5-I24-A21.allc.tsv.gz.tbi
│   ├── FC_M_E15a_3C_1-1-I5-B1.allc.tsv.gz
│   ├── FC_M_E15a_3C_1-1-I5-B1.allc.tsv.gz.tbi
│   ├── FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz
│   ├── FC_M_P12b_3C_2-5-M17-N10.allc.tsv.gz.tbi
│   ├── FC_M_P3b_3C_6-6-J3-P24.allc.tsv.gz
│   ├── FC_M_P3b_3C_6-6-J3-P24.allc.tsv.gz.tbi
│   ├── FC_M_P6a_3C_7-3-K21-P5.allc.tsv.gz
│   ├── FC_M_P6a_3C_7-3-K21-P5.allc.tsv.gz.tbi
│   ├── FC_M_P9B_3C_6-2-F6-O4.allc.tsv.gz
│   ├── FC_M_P9B_3C_6-2-F6-O4.allc.tsv.gz.tbi
│   ├── FC_P0b_3C_5-1-I24-J14.allc.tsv.gz
│   ├── FC_P0b_3C_5-1-I24-J14.allc.tsv.gz.tbi
│   ├── FC_P13a_3C_7-1-A11-O1.allc.tsv.gz
│   ├── FC_P13a_3C_7-1-A11-O1.allc.tsv.gz.tbi
│   ├── FC_P28a_3C_2-1-E5-N14.allc.tsv.gz
│   └── FC_P28a_3C_2-1-E5-N14.allc.tsv.gz.tbi
├── all_cells.cz
├── allc_to_cz_benchmark
│   └── allc_to_cz_benchmark.tsv
├── bam_benchmark
│   ├── bam_benchmark.tsv
│   └── bam_benchmark.txt
├── bg_gene.h5ad
├── cz
│   ├── F